In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np 
import time 
from sqlalchemy import create_engine
from pathlib import Path

In [52]:
def get_year_draft_results(year):
    url = f"https://www.basketball-reference.com/draft/{"NBA" if year >= 1950 else "BAA"}_{year}.html"

    df = pd.read_html(url, attrs={"id" : "stats"})[0]
    df.columns = [i[1] for i in df.columns.to_flat_index()]

    df = df[["Rk", "Pk", "Tm", "Player", "College"]].copy()
    df["Rk"] = pd.to_numeric(df["Rk"], errors="coerce")
    df = df[df.Rk.notna() & df.Player.notna()]
    df["College"] = df["College"].fillna("None")

    return df

def get_selected_years_draft_results(years, page_limit):
    rate_limiting = len(years) >= page_limit
    
    if rate_limiting: 
        pages_visited = 0
        start_time = time.time()
    
    df = pd.DataFrame()
    for year in years:
        if rate_limiting and pages_visited == page_limit:
            current_time = time.time()
            wait_time = max(0, 60 - (current_time - start_time))
            print(f"Rate limited. Waiting {wait_time:.2f} seconds")

            time.sleep(max(0, 60 - (current_time - start_time)))
            start_time = time.time()
            pages_visited = 0

        if year < 1947 or year > 2026:
            print(f"Year is invalid. Skipping {year}...")
            continue 

        year_df = get_year_draft_results(year)
        
        if rate_limiting:
            pages_visited += 1

        year_df["Year"] = year
        
        df = pd.concat([df, year_df], axis=0)

        print(f"Draft history added for year: {year}")

    return df

In [53]:
df = get_selected_years_draft_results(list(range(1947, 2026)), 19)

Draft history added for year: 1947
Draft history added for year: 1948
Draft history added for year: 1949
Draft history added for year: 1950
Draft history added for year: 1951
Draft history added for year: 1952
Draft history added for year: 1953
Draft history added for year: 1954
Draft history added for year: 1955
Draft history added for year: 1956
Draft history added for year: 1957
Draft history added for year: 1958
Draft history added for year: 1959
Draft history added for year: 1960
Draft history added for year: 1961
Draft history added for year: 1962
Draft history added for year: 1963
Draft history added for year: 1964
Draft history added for year: 1965
Rate limited. Waiting 53.22 seconds
Draft history added for year: 1966
Draft history added for year: 1967
Draft history added for year: 1968
Draft history added for year: 1969
Draft history added for year: 1970
Draft history added for year: 1971
Draft history added for year: 1972
Draft history added for year: 1973
Draft history added

In [62]:
def move_draft_history_to_database(draft_history):
    db_path = Path("~/Personal Project/data/nba.db").expanduser()
    engine = create_engine(f"sqlite:///{db_path}")
    
    draft_history.to_sql(
        "draft_history",
        engine,
        if_exists="append",
        index=False
    )

    print("Successfully moved to database.")

move_draft_history_to_database(df)

Successfully moved to database.


In [2]:
db_path = Path("~/Personal Project/data/nba.db").expanduser()
engine = create_engine(f"sqlite:///{db_path}")

pd.read_sql("SELECT * FROM draft_history", engine)

,Rk,Pk,Tm,Player,College,Year
0,1.0,1,PIT,Clifton McNeely,Texas Wesleyan University,1947
1,2.0,2,TRH,Glen Selbo,Wisconsin,1947
2,3.0,3,BOS,Bulbs Ehlers,Purdue,1947
3,4.0,4,PRO,Walt Dropo,UConn,1947
4,5.0,5,NYK,Dick Holub,Long Island University,1947
...,...,...,...,...,...,...
8408,26.0,26,DEN,Tarris Reed Jr.,UConn,2026
8409,27.0,27,BOS,Chris Cenac Jr.,Houston,2026
8410,28.0,28,MIN,Joshua Jefferson,Iowa State,2026
8411,29.0,29,CLE,Alex Karaban,UConn,2026
